# 🏦 Blueprint: Análise Profunda do Setor Financeiro (B3)

Este notebook implementa a camada de **Analytics Avançado** para o setor financeiro, utilizando os dados processados do **Master Data Lake 3.1**.

### 📋 Camadas Analíticas Implementadas:
1. **Performance e Valorização:** Retornos acumulados e tendências.
2. **Risco e Resiliência:** Sharpe Ratio, Volatilidade e Drawdown.
3. **Eficiência e Fundamentos:** ROE, Payout Ratio e Market Cap.
4. **Matriz de Correlação:** Interdependência entre grandes bancos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os
from IPython.display import display

# Configurações estéticas
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Caminho para o setor financeiro (Parquet Particionado)
caminho_setor = '../data/processed/setores/Financial_Services.parquet'
df = pd.read_parquet(caminho_setor)

print(f"📦 Dataset do Setor Financeiro carregado: {len(df)} registros.")
print(f"🏢 Ativos encontrados: {df['ticker'].unique()}")
df['date'] = pd.to_datetime(df['date'])

## 1️⃣ Dashboard de Performance: Retorno Acumulado
Comparação direta da valorização de longo prazo entre os principais bancos.

In [ ]:
# Filtramos apenas tickers relevantes (Ex: ITUB4, BBAS3, BBDC, SANB)
ativos_comp = ['ITUB4.SA', 'BBAS3.SA', 'BBDC4.SA', 'SANB11.SA', 'B3SA3.SA']
df_comp = df[df['ticker'].isin(ativos_comp)].copy()

fig = px.line(df_comp, x='date', y='retorno_acumulado', color='ticker', 
              title='📈 Performance Acumulada: Setor Financeiro',
              labels={'retorno_acumulado': 'Crescimento (%)'})
fig.show()

## 2️⃣ Dashboard de Risco: Sharpe vs Volatilidade
Onde o investidor encontra o melhor equilíbrio entre retorno e oscilação?

In [ ]:
# Médias de risco no período
df_risco = df_comp.groupby('ticker').agg({
    'sharpe_ratio_21d': 'mean', 
    'volatilidade_252d': 'mean',
    'drawdown': 'min'
}).reset_index()

fig = px.scatter(df_risco, x='volatilidade_252d', y='sharpe_ratio_21d', 
                 size=np.abs(df_risco['drawdown']), text='ticker', color='ticker',
                 title='🎯 Scatter Plot: Risco vs Retorno (Tamanho do círculo = Máx Queda)',
                 labels={'volatilidade_252d': 'Risco (Volatilidade)', 'sharpe_ratio_21d': 'Relação Retorno/Risco'})
fig.show()

## 3️⃣ Dashboard de Eficiência: ROE vs Payout vs Market Cap
Analisa a lucratividade vs quanto de lucro é distribuído.

In [ ]:
df_latest = df_comp.sort_values('date').groupby('ticker').tail(1)

fig = px.bar(df_latest, x='ticker', y='roe', color='payout_ratio', 
             title='💎 Eficiência: ROE vs Payout Ratio (Cores)',
             labels={'roe': 'ROE (Rentabilidade)', 'payout_ratio': 'Taxa de Payout'})
fig.show()

print("📊 Tendência de Dividend Yield (DY):")
display(df_latest[['ticker', 'dy_ltm', 'tendencia_dy_3y']])

## 4️⃣ Matriz de Correlação: Interdependência
Se um banco cai, todos caem? Vamos ver o Heatmap de retornos logarítmicos.

In [ ]:
df_pivot = df_comp.pivot(index='date', columns='ticker', values='log_return').dropna()
corr = df_pivot.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('🔥 Matriz de Correlação: Grandes Instituições Financeiras')
plt.show()